In [27]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

### Adquisición y tratamiento de datos

In [28]:
# Raíz del repositorio: primera carpeta (desde la actual hacia arriba) que contiene .git
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.git').exists())
carpeta_sinca = raiz / 'Datos' / 'Datos_SINCA'
carpeta_meteo = raiz / 'Datos' / 'Open_Meteo' / 'Previous_Runs'

# El SINCA registra en hora local estándar de Chile (UTC-4) todo el año, sin horario de verano: su grilla horaria
# no tiene horas duplicadas ni faltantes, y su ciclo diario de temperatura coincide con el de Open-Meteo
# desplazado 4 h tanto en verano como en invierno. Open-Meteo viene en UTC, por lo que se lleva a UTC-4.
DESFASE_SINCA_UTC = pd.Timedelta(hours=-4)

def leer_sinca(archivo):
    df = pd.read_csv(
        archivo, sep=';', decimal=',',
        dtype={'FECHA (YYMMDD)': str, 'HORA (HHMM)': str},
    ).dropna(axis=1, how='all')
    df['fecha_hora'] = pd.to_datetime(df['FECHA (YYMMDD)'] + df['HORA (HHMM)'], format='%y%m%d%H%M')
    return df.drop(columns=['FECHA (YYMMDD)', 'HORA (HHMM)']).set_index('fecha_hora')

def leer_open_meteo(archivo):
    # Pronóstico a D+1 (corrida del día anterior). Las columnas se renombran con el prefijo 'om_'
    # para distinguirlas de la meteorología medida por el SINCA (ej: om_temperature_2m vs Temperatura (C)).
    df = pd.read_csv(archivo, parse_dates=['fecha_hora_utc'], index_col='fecha_hora_utc')
    df.index = (df.index + DESFASE_SINCA_UTC).rename('fecha_hora')
    return df.rename(columns=lambda c: 'om_' + c.removesuffix('_previous_day1'))

# Cada estación del SINCA se une con el Open-Meteo de sus propias coordenadas. Open-Meteo es la fuente más
# limitada (desde enero de 2024), así que el cruce es interno: solo se conservan las horas presentes en ambas
# fuentes, para que toda fila disponible para el modelo tenga los predictores meteorológicos.
# Los extractores descargan las mismas estaciones (las que miden MP2.5 vigente desde 2024), por lo que cada
# archivo del SINCA tiene su par en Open-Meteo.
dfs = {
    archivo.stem: leer_sinca(archivo).join(leer_open_meteo(carpeta_meteo / archivo.name), how='inner')
    for archivo in sorted(carpeta_sinca.glob('*.csv'))
}

print(f"Período común SINCA + Open-Meteo: {min(df.index.min() for df in dfs.values())} -> "
      f"{max(df.index.max() for df in dfs.values())}")

Período común SINCA + Open-Meteo: 2024-01-19 08:00:00 -> 2026-11-16 19:00:00


In [29]:
#Se eliminan los registros preliminares y no validados de todos los contaminantes (MP2.5, PM10, NO, NO2, NOx, O3, CO, SO2),
#a modo de trabajar solo con registros validados y finales por hora.
#Luego se marcan como NaN los valores no numéricos y los físicamente imposibles (fallas o valores de relleno del
#sensor, p. ej. -40 °C o 2.7e23 m/s). Se anula solo el valor inválido y no la fila completa, para no perder
#mediciones válidas de otras variables en esa hora.
#Finalmente se eliminan las columnas sin ningún dato en el período (p. ej. SO2 en estaciones que no lo miden desde 2024)
#y las filas sin ningún registro del SINCA (las columnas 'om_' de Open-Meteo no cuentan: una hora con solo
#pronóstico y sin mediciones no aporta).
rangos_validos = {
    'Temperatura (C)': (-10, 45),
    'Humedad relativa (%)': (0, 100),
    'Velocidad viento (m/s)': (0, 30),
    'Direccion viento (grados)': (0, 360),
}
for nombre in list(dfs):
    df = dfs[nombre]
    filas_antes = len(df)
    no_validadas = [c for c in df.columns if 'Registros preliminares' in c or 'Registros no validados' in c]
    df = df.drop(columns=no_validadas)

    # Un texto en una columna numérica haría fallar between(); se convierte a NaN y se informa
    numerico = df.apply(pd.to_numeric, errors='coerce')
    invalidos = {f'{c} (no numérico)': n for c, n in (numerico.isna() & df.notna()).sum().items() if n}
    df = numerico

    for col, (minimo, maximo) in rangos_validos.items():
        if col in df:
            fuera = df[col].notna() & ~df[col].between(minimo, maximo)
            if fuera.any():
                invalidos[col] = int(fuera.sum())
                df[col] = df[col].mask(fuera)

    vacias = df.columns[df.isna().all()].tolist()
    df = df.drop(columns=vacias)
    columnas_sinca = [c for c in df.columns if not c.startswith('om_')]
    df = df.dropna(how='all', subset=columnas_sinca) if columnas_sinca else df.iloc[0:0]

    if df.empty:
        print(f"{nombre}: sin registros válidos del SINCA, se descarta")
        del dfs[nombre]
        continue
    dfs[nombre] = df
    print(f"{nombre}: {filas_antes - len(df)} filas eliminadas | valores inválidos anulados: {invalidos or 'ninguno'}"
          + (f" | columnas vacías eliminadas: {vacias}" if vacias else ""))

Cerrillos_II: 2149 filas eliminadas | valores inválidos anulados: ninguno
Cerro_Navia: 56 filas eliminadas | valores inválidos anulados: ninguno | columnas vacías eliminadas: ['SO2 - Registros validados']
El_Bosque: 10231 filas eliminadas | valores inválidos anulados: ninguno
La_Florida: 109 filas eliminadas | valores inválidos anulados: {'Velocidad viento (m/s)': 3, 'Direccion viento (grados)': 3}
Las_Condes: 1891 filas eliminadas | valores inválidos anulados: ninguno | columnas vacías eliminadas: ['SO2 - Registros validados']
Parque_OHiggins: 162 filas eliminadas | valores inválidos anulados: ninguno | columnas vacías eliminadas: ['SO2 - Registros validados']
Pudahuel: 289 filas eliminadas | valores inválidos anulados: ninguno | columnas vacías eliminadas: ['SO2 - Registros validados']
Puente_Alto: 222 filas eliminadas | valores inválidos anulados: ninguno | columnas vacías eliminadas: ['SO2 - Registros validados']
Quilicura: 262 filas eliminadas | valores inválidos anulados: ninguno